# Data Cleaning: Impacts of Data Centers on Electrical Utilities

This notebook loads the raw data that `01-data_collection.ipynb` saved to `data/raw/` and cleans it.

The main output is the cleaned FracTracker data centers table with an `is_crypto` label.

The EIA sections near the end are a first look only. They are not saved yet.

In [ ]:
import re
import pandas as pd
import numpy as np
from pathlib import Path
import geocoder
from geopy.distance import geodesic

In [ ]:
DATA_DIR = Path("data/raw")

# display options for pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## First look at each dataset

### FracTracker U.S. Data Centers

In [ ]:
data_centers = pd.read_csv(DATA_DIR / "fractracker_data_centers.csv")
print("Rows and columns:", data_centers.shape)
data_centers.sample(3)

In [ ]:
# Summary Statistics for Data Centers Dataset
# Looking for anything that needs to be cleaned
data_centers.describe(include="all").T

### FracTracker U.S. Data Centers - Data Cleaning

| Feature | Identified Issue | Proposed Solution |
|---|---|---|
| `mw` | Contains both numerical values and ranges. | Separate ranges into `mw_low` and `mw_high` to preserve the available information. |
| `power_source` | String categorical values. | Standardize categories and encode them for modeling. |
| `dedicated_power_plant` | Limited availability across facilities. | Determine if power plants can be identified using location (latitude/longitude) and power source. |
| `number_of_generators` | Contains numerical information stored as strings; limited availability across facilities. | Too limited to be a learnable feature unless more information can be added. |
| `number_of_buildings` | Contains integers, ranges, and strings (e.g., `6+`, `1-19`, `8`, `3 large, 1 small`). | Create a consistent approach for representing ranges and descriptive counts. |
| `cooling_source` | Categorical values stored as strings. | Standardize categories and encode them for modeling. |
| `cooling_type` | Inconsistent categorical formatting (e.g., `Closed loop` vs. `Closed Loop`). | Normalize categorical values and encode them for modeling. |
| `project_cost` | Contains strings and potentially ranges rather than consistent numerical values. | Extract numerical values and determine an appropriate method for representing ranges. |



`location_confidence`

In [ ]:
# Small fix for location confidence values
data_centers['location_confidence'] = (
    data_centers['location_confidence']
    .apply(lambda x: 'High' if x == 'high' else x)
)

`sizerank`

In [ ]:
# The Mega campus category is spelled two ways
data_centers["sizerank"] = data_centers["sizerank"].replace(
    "Mega Campus (>1,000 MW)", "Mega campus (>1,000 MW)"
)

data_centers["sizerank"].value_counts(dropna=False)

`mw`

In [ ]:
# Remove commas and spaces
mw = data_centers['mw'].str.replace(',', '').str.strip()

# Split and assign ranges
data_centers[['mw_low', 'mw_high']] = mw.str.split('-', expand=True)
data_centers['mw_high'] = data_centers['mw_high'].fillna(data_centers['mw_low']) # fill where no range exists 

# Convert to numeric
data_centers[['mw_low', 'mw_high']] = (
    data_centers[['mw_low', 'mw_high']]
    .apply(pd.to_numeric, errors='coerce')
)

data_centers.loc[data_centers['mw'].notna()][['mw', 'mw_low', 'mw_high']].head()

`power_source`

In [ ]:
# Normalize power sources
power_source = (
    data_centers['power_source']
    .str.lower()
    .str.strip()
    .str.replace('(', '')
    .str.replace(')', '')
    .str.replace('/', '_')
    .str.replace(' ', '_')
)

# One-hot encoding for each source type
source_encoded = power_source.str.get_dummies(sep=",_")
data_centers = pd.concat([data_centers, source_encoded], axis=1)

columns = [
    "power_source", "batteries", "biomass", "coal", "fuel_cells", 
    "geothermal", "grid_unspecified_mix", "hybrid_onsite_and_grid", 
    "hydroelectric", "natural_gas", "nuclear", "oil_diesel", 
    "renewable", "solar", "wind"
]

data_centers.loc[data_centers['power_source'].notna()][columns].sample(5)

`number_of_buildings`

In [ ]:
# Standardization function for assigning number of buildings
def standardize_buildings(buildings):
    if pd.isna(buildings):
        return np.nan, np.nan
    
    buildings = buildings.replace(' ', '')
    b_low = np.nan
    b_high = np.nan
    
    if '-' in buildings:
        split = buildings.split('-')
        b_low = int(split[0])
        b_high = int(split[1])
        
    elif '+' in buildings:
        buildings = buildings.replace('+', '')
        b_low = int(buildings)
        
    elif buildings.isdigit():
        b_low = int(buildings)
        b_high = int(buildings)
        
    return b_low, b_high
    
# Add new features to handle ranges
data_centers[["buildings_low", "buildings_high"]] = pd.DataFrame(
    data_centers["number_of_buildings"].apply(standardize_buildings).tolist(),
    index=data_centers.index
)

data_centers.loc[data_centers["buildings_low"].notna()][["number_of_buildings", "buildings_low", "buildings_high"]].head()

`cooling_source` + `cooling_type`

In [ ]:
for col in ['cooling_source', 'cooling_type']:
    # Normalize cooling cols
    cooling_data = (
        data_centers[col]
        .str.lower()
        .str.strip()
        .str.replace('/', '_')
        .str.replace(' ', '_')
    )

    # One-hot encoding for each source type
    source_encoded = cooling_data.str.get_dummies()
    data_centers = pd.concat([data_centers, source_encoded], axis=1)

columns = ['cooling_source', 'air', 'water', 'hybrid_air_water']

data_centers.loc[data_centers['cooling_source'].notna()][columns].sample(5)

In [ ]:
columns = ['cooling_type', 'closed_loop', 'fans', 'open_loop']

data_centers.loc[data_centers['cooling_type'].notna()][columns].sample(5)

`project_cost`

In [ ]:
def standardize_project_cost(cost):
    if pd.isna(cost):
        return np.nan, np.nan
    
    multiplier = 1
    cost_low = np.nan
    cost_high = np.nan
    
    # Normalize cost
    cost = (
        str(cost)
        .lower()
        .replace(' ', '')
        .replace('$', '')
        .replace(',', '')
        .replace('miillion', 'million')
    )

    # Replace abbreviated m
    cost = re.sub(r'm$', 'million', cost)
    
    if 'million' in cost:
        multiplier = 1000000
    elif 'billion' in cost:
        multiplier = 1000000000
        
    cost = re.sub(r'[mb]illion', '', cost)
    
    # Assign range to low and high
    if '-' in cost:
        split = cost.split('-')
        cost_low = float(split[0]) * multiplier
        cost_high = float(split[1]) * multiplier
        
    elif '+' in cost:
        cost = cost.replace('+', '')
        cost_low = float(cost) * multiplier
        
    else:
        cost_low = float(cost) * multiplier
        cost_high = float(cost) * multiplier
        
    return cost_low, cost_high

# Add new features for handling the cost range
data_centers[["project_cost_low", "project_cost_high"]] = pd.DataFrame(
    data_centers["project_cost"].apply(standardize_project_cost).tolist(),
    index=data_centers.index
)

data_centers[data_centers["project_cost"].notna()][["project_cost", "project_cost_low", "project_cost_high"]].sample(5)

### FracTracker Crypto Mining Operations

In [ ]:
crypto = pd.read_csv(DATA_DIR / "fractracker_crypto.csv")
print("Rows and columns:", crypto.shape)
crypto.sample(3)

In [ ]:
crypto.describe(include="all").T

### FracTracker Crypto Mining Operations - Data Cleaning

| Feature | Identified Issue | Proposed Solution |
|---|---|---|
| `Probable power source` | Power source values use inconsistent naming conventions. | Normalize them to the FracTracker data center categories using keyword mapping. |
| `Precise Location?` | Values represented as 'Yes' / 'No' strings. | Update to a boolean representation. |
| `MW (NYTimes)` | A range of values including '100 MW', '100 MW+', '100 MW --> 200 MW'. | Standardize to `mw_low` and `mw_high` to keep consistent representation in both datasets. |

`Probable power source`

In [ ]:
# Reuse power sources from FracTracker data centers
power_sources = [
    "batteries", "biomass", "coal", "fuel_cells",
    "geothermal", "grid_unspecified_mix", "hybrid_onsite_and_grid",
    "hydroelectric", "natural_gas", "nuclear", "oil_diesel",
    "renewable", "solar", "wind"
]

# Mapping given sources to predefined power sources
def normalize_sources(source):
    if pd.isna(source):
        return []

    source = source.lower()
    found_sources = set()

    # Keyword mappings
    keyword_map = {
        "battery": "batteries",
        "biomass": "biomass",
        "coal": "coal",
        "fuel cell": "fuel_cells",
        "geothermal": "geothermal",
        "hydro": "hydroelectric",
        "gas": "natural_gas",
        "nuclear": "nuclear",
        "diesel": "oil_diesel",
        "oil": "oil_diesel",
        "solar": "solar",
        "wind": "wind",
        "renewable": "renewable",
        "wte": "renewable",
    }

    for keyword, category in keyword_map.items():
        if keyword in source:
            found_sources.add(category)

    # Grid
    if "grid" in source:
        found_sources.add("grid_unspecified_mix")

    # Grid + another identified source = hybrid
    non_grid_sources = found_sources - {"grid_unspecified_mix"}

    if "grid_unspecified_mix" in found_sources and non_grid_sources:
        found_sources.add("hybrid_onsite_and_grid")

    return sorted(found_sources)


crypto["norm_power_source"] = crypto["Probable power source"].apply(normalize_sources)
crypto.loc[crypto["norm_power_source"].str.len() > 0][["norm_power_source"]].sample(5)

`Precise Location?`

In [ ]:
# Change precise location to a boolean
crypto["Precise Location?"] = crypto["Precise Location?"].apply(lambda x: x == 'Yes')
crypto[['Precise Location?']].sample(5)

`MW (NYTimes)`

In [ ]:
# Function for standardizing mw values
def standardize_mw(value):
    if pd.isna(value):
        return np.nan, np.nan

    text = str(value).lower().strip()

    # Fix typo
    text = text.replace("nw", "mw")

    # Extract numbers followed by MW or GW
    matches = re.findall(r'(\d+(?:\.\d+)?)\s*(mw|gw)', text)

    if not matches:
        return np.nan, np.nan

    values_mw = []

    for number, unit in matches:
        number = float(number)

        # Convert to MW
        if unit == "gw":
            number *= 1000

        values_mw.append(number)

    return min(values_mw), max(values_mw)


# Add new features to handle ranges
crypto[["mw_low", "mw_high"]] = pd.DataFrame(
    crypto["MW (NYTimes)"].apply(standardize_mw).tolist(),
    index=crypto.index
)

crypto.loc[crypto['MW (NYTimes)'].notna()][["MW (NYTimes)", "mw_low", "mw_high"]].sample(5)

### Address Standardization & Comparison 

Addresses are standardized across both datasets to make facility matching more reliable. Common street names, directions, and unit types are converted to consistent abbreviations, while special characters and extra whitespace are removed. The crypto dataset sometimes stores the city, state, and ZIP code inside the Address field. These values are separated so the final location fields are: 

- **Crypto**: `Address`, `City`, `State`, `Zip` 
- **Data Centers**: `address`, `city`, `state`, `zip`

#### Address Abbreviations

In [ ]:
ADDRESS_ABBREVIATIONS = {
    # Directions
    'north': 'n',
    'south': 's',
    'east': 'e',
    'west': 'w',
    'northeast': 'ne',
    'northwest': 'nw',
    'southeast': 'se',
    'southwest': 'sw',

    # Street suffixes
    'street': 'st',
    'avenue': 'ave',
    'road': 'rd',
    'drive': 'dr',
    'lane': 'ln',
    'boulevard': 'blvd',
    'circle': 'cir',
    'court': 'ct',
    'place': 'pl',
    'parkway': 'pkwy',
    'highway': 'hwy',
    'trail': 'trl',
    'terrace': 'ter',
    'square': 'sq',
    'crossing': 'xing',
    'center': 'ctr',
    'centre': 'ctr',
    'point': 'pt',
    'ridge': 'rdg',
    'run': 'run',
    'view': 'vw',
    'village': 'vlg',
    'station': 'sta',
    'landing': 'lndg',
    'junction': 'jct',
    'extension': 'ext',
    'expressway': 'expy',
    'freeway': 'fwy',
    'turnpike': 'tpke',
    'causeway': 'cswy',
    'crescent': 'cres',
    'cove': 'cv',
    'fork': 'frk',
    'garden': 'gdn',
    'gardens': 'gdns',
    'grove': 'grv',
    'harbor': 'hbr',
    'harbour': 'hbr',
    'heights': 'hts',
    'hill': 'hl',
    'hills': 'hls',
    'island': 'is',
    'islands': 'iss',
    'knoll': 'knl',
    'knolls': 'knls',
    'lake': 'lk',
    'lakes': 'lks',
    'manor': 'mnr',
    'manors': 'mnrs',
    'meadow': 'mdw',
    'meadows': 'mdws',
    'mount': 'mt',
    'mountain': 'mtn',
    'orchard': 'orch',
    'pine': 'pne',
    'pines': 'pnes',
    'plaza': 'plz',
    'port': 'prt',
    'prairie': 'pr',
    'radial': 'radl',
    'river': 'riv',
    'shore': 'shr',
    'shores': 'shrs',
    'spring': 'spg',
    'springs': 'spgs',
    'summit': 'smt',
    'union': 'un',
    'valley': 'vly',
    'valleys': 'vlys',
    'well': 'wl',
    'wells': 'wls',
    
    # Unit Overrides
    'apartment': 'apt',
    'apt.': 'apt',
    'suite': 'ste',
    'ste.': 'ste',
    'unit': 'unit',
    '#': 'unit',
    'building': 'bldg',
    'floor': 'fl',
    'room': 'rm',
    'department': 'dept'
}

#### Helper Functions

These functions are used to standardize addresses by removing special characters and extra whitespace and replacing common address terms with predefined abbreviations. In the crypto dataset, the `Address` field sometimes also contains the city, state, and ZIP code. A second function, `remove_city_state`, removes this duplicated location information, while ZIP codes are extracted into a separate column.

In [ ]:
def standardize_address(address):
    if pd.isna(address):
        return np.nan
    
    address = address.lower().strip()
    
    # Remove special characters
    address = re.sub(r'[^a-z0-9\s]', '', address)
    address_parts = address.split()
    
    # Update addresses using abbreviations
    cleaned_parts = [
        ADDRESS_ABBREVIATIONS.get(part, part)
        for part in address_parts
    ]
    
    return " ".join(cleaned_parts)


def remove_city_state(address, city, state):
    if pd.isna(address):
        return np.nan

    address = str(address)

    # Remove ZIP from address
    address = re.sub(r'\s*,?\s*\d{5}\s*$', '', address)

    # Remove State from address
    if pd.notna(state):
        address = re.sub(
            rf'\s*,?\s*{re.escape(str(state))}\s*$',
            '',
            address,
            flags=re.IGNORECASE
        )

    # Remove City from address
    if pd.notna(city):
        address = re.sub(
            rf'\s*,?\s*{re.escape(str(city))}\s*$',
            '',
            address,
            flags=re.IGNORECASE
        )

    return address.strip(' ,')

In [ ]:
# Extract 5-digit zip if exists
crypto['Zip'] = crypto['Address'].str.extract(r'(\d{5})\s*$')[0]
crypto['Zip'] = crypto['Zip'].replace('', np.nan).astype('Int64') # missing zip

# Remove city, state, and zip from addresses in crypto
crypto['Address'] = crypto.apply(
    lambda row: remove_city_state(
        row['Address'],
        row['City'],
        row['State']
    ),
    axis=1
)

# Applying the transformations to the addresses in both datasets
crypto['Address'] = crypto['Address'].apply(standardize_address)
data_centers['address'] = data_centers['address'].apply(standardize_address)


# data_centers[['address', 'city', 'state', 'zip']].sample(5)
crypto[['Address', 'City', 'State', 'Zip']].sample(5)

#### Address Comparison

The standardized datasets are merged using exact matches on address, city, and state to identify facilities that appear in both datasets. This allows us to identify data centers associated with known crypto mining locations, providing more information for later analysis. Using exact address matching, 19 matching records were identified between the two datasets.

In [ ]:
# Find matching addresses between the datasets
matches = crypto[crypto['Address'].notna()].merge(
    data_centers[data_centers['address'].notna()],
    left_on=['Address', 'City', 'State'],
    right_on=['address', 'city', 'state'],
    how='inner',
    suffixes=('_crypto', '_dc')
)

print(f"{len(matches)} matches found between the crypto and data centers datasets.")
matches[['facility_name', 'Name', 'lat_crypto', 'long_crypto', 'lat_dc', 'long_dc', 'address', 'city', 'state']]

#### Crypto Label for Data Centers

The FracTracker data center list does not say which facilities are crypto mines. We build an `is_crypto` label here so `03-supervised-learning.ipynb` can load it from `data/processed/`.

The label comes from **two different sources**:

1. **The FracTracker crypto dataset.** A data center is crypto if it matches a known crypto mining site, either:
   - by location: a crypto site is within `MAX_MILES` of it, or
   - by exact address (the Address Comparison above).
2. **The FracTracker data centers dataset itself.** Some data centers already say they are crypto in their `purpose` or `facility_name` (for example "Bitcoin" or "Crypto mining/AI"). Many of these are not in the crypto dataset at all.

We keep one column for each check (`crypto_from_distance`, `crypto_from_address`, `crypto_from_dc_dataset`) so we can always see where a label came from.

##### How old is each dataset?

The crypto dataset is older than the data centers dataset:

- **Crypto sites:** the `Date` column runs from March 2023 to April 2026, but most entries are from 2023 and 2024 (median: May 2024).
- **Data centers:** the `date_updated` column runs from January 2025 to September 2026 (median: November 2025).

So there is roughly a year and a half between the typical crypto entry and the typical data center entry. This matters for the crypto label:

- A data center that matched a crypto site tells us the site **was** crypto when the crypto dataset was made. It may not still be crypto. Several sites are converting from crypto mining to AI (for example Applied Digital, Crexi and Keel Infrastructure).
- A "Proposed" crypto site from 2023 may never have been built, or may have been built as a different kind of data center.
- The `crypto_from_dc_dataset` column comes from the newer dataset, so it is the more up-to-date of the two sources.

In [ ]:
# Compare how old each dataset is
crypto_dates = pd.to_datetime(crypto["Date"], format="%m/%d/%Y", errors="coerce")
dc_dates = pd.to_datetime(data_centers["date_updated"], unit="ms")

print("Crypto sites with a date:", crypto_dates.notna().sum(), "of", len(crypto))
print("Crypto dates:            ", crypto_dates.min().date(), "to", crypto_dates.max().date())
print("Crypto median date:      ", crypto_dates.median().date())
print()
print("Data centers last updated:", dc_dates.min().date(), "to", dc_dates.max().date())
print("Data centers median date: ", dc_dates.median().date())
print()
print("Crypto sites by year:")
print(crypto_dates.dt.year.value_counts().sort_index())
print()
print("Data centers by year last updated:")
print(dc_dates.dt.year.value_counts().sort_index())

##### Drop invalid coordinates

A few rows have coordinates that cannot be real: a longitude of `-11402.57827` (likely a typo) and a data center at `0, 0` (missing location). We can't measure distance for these, so we drop them.

In [ ]:
def valid_coordinates(df):
    return df["lat"].between(-90, 90) & df["long"].between(-180, 180) & ~((df["lat"] == 0) & (df["long"] == 0))

print("Dropping data centers:", data_centers.loc[~valid_coordinates(data_centers), "facility_name"].tolist())
print("Dropping crypto sites:", crypto.loc[~valid_coordinates(crypto), "Name"].tolist())

data_centers = data_centers[valid_coordinates(data_centers)].reset_index(drop=True)
crypto = crypto[valid_coordinates(crypto)].reset_index(drop=True)

##### Distance between every crypto site and every data center

`geodesic` from the `geopy` package takes two `(lat, long)` points and returns the distance between them along the Earth's surface.

We loop over every crypto site and every data center, which is about 500,000 pairs, so this cell takes around 30 seconds.

The result is a table with one row per crypto site and one column per data center.

In [ ]:
crypto_points = list(zip(crypto["lat"], crypto["long"]))
dc_points = list(zip(data_centers["lat"], data_centers["long"]))

distances = []
for crypto_point in crypto_points:
    row = []
    for dc_point in dc_points:
        row.append(geodesic(crypto_point, dc_point).miles)
    distances.append(row)

# turn the list of rows into a table numpy can search
distances = np.array(distances)

print("Distance table (crypto sites, data centers):", distances.shape)

##### Nearest data center for each crypto site

Before picking a cutoff, look at how far each crypto site is from its closest data center.

In [ ]:
nearest_dc = distances.argmin(axis=1)

crypto["nearest_dc_miles"] = distances.min(axis=1)
crypto["nearest_dc_name"] = data_centers["facility_name"].values[nearest_dc]
crypto["nearest_dc_operator"] = data_centers["operator_name"].values[nearest_dc]

for miles in [0.1, 0.25, 0.5, 1, 2, 5]:
    # keep only the crypto sites that are close enough, then count them
    nearby = crypto[crypto["nearest_dc_miles"] <= miles]
    print(f"Crypto sites with a data center within {miles} mi: {len(nearby)}")

Most crypto sites have no data center nearby. Many crypto mines are simply not in the data center list.

The table below puts the names side by side so we can check whether close pairs are really the same facility.

In [ ]:
columns = [
    "Name", "Company", "Precise Location?",
    "nearest_dc_name", "nearest_dc_operator", "nearest_dc_miles"
]

crypto.loc[crypto["nearest_dc_miles"] <= 2, columns].sort_values("nearest_dc_miles")

##### Source 1: the crypto dataset

We label from the data center side: for each data center, find the closest crypto site. If it is within `MAX_MILES`, `crypto_from_distance` is True.

Labeling this way handles two things:
- The crypto file repeats some sites (same name or same coordinates), and several crypto rows can point to one data center.
- Every data center gets exactly one label.

`0.5` miles is a starting point. Some sites under 2 miles are clearly different facilities, for example CleanSpark (Clinton) and Amazon Clinton Data Center.

We also keep the exact address matches from above. Two of them are real matches that are a little more than 0.5 miles apart (Helios Data Center and Riot Platforms Rockdale).

In [ ]:
MAX_MILES = 0.5

nearest_crypto = distances.argmin(axis=0)

data_centers["crypto_miles"] = distances.min(axis=0)
data_centers["crypto_name"] = crypto["Name"].values[nearest_crypto]
data_centers["crypto_company"] = crypto["Company"].values[nearest_crypto]
data_centers["crypto_status"] = crypto["Status"].values[nearest_crypto]

# Check 1: a crypto site is within MAX_MILES of the data center
data_centers["crypto_from_distance"] = data_centers["crypto_miles"] <= MAX_MILES

# Check 2: the data center matched a crypto site by exact address
address_match_ids = list(matches["facility_id"])
data_centers["crypto_from_address"] = data_centers["facility_id"].isin(address_match_ids)

print("Crypto from distance:", data_centers["crypto_from_distance"].sum())
print("Crypto from address: ", data_centers["crypto_from_address"].sum())

##### Source 2: the data centers dataset

We search `purpose` and `facility_name` for words that mean crypto mining. We do not search for "crypto" everywhere, because it would also match "NSA Texas Cryptology Center".

In [ ]:
# "^crypto" means the text starts with "crypto"
crypto_words = "bitcoin|crypto mining|crypto mine|cryptomine|cryptocurrency|^crypto"

purpose_says_crypto = data_centers["purpose"].str.contains(crypto_words, case=False, na=False)
name_says_crypto = data_centers["facility_name"].str.contains(crypto_words, case=False, na=False)

# Check 3: the data center says it is crypto
data_centers["crypto_from_dc_dataset"] = purpose_says_crypto | name_says_crypto

print("Crypto from purpose or name:", data_centers["crypto_from_dc_dataset"].sum())

##### Final label

A data center gets `is_crypto = 1` if any of the three checks is True.

In [ ]:
matched_crypto_site = data_centers["crypto_from_distance"] | data_centers["crypto_from_address"]

data_centers["is_crypto"] = (
    matched_crypto_site | data_centers["crypto_from_dc_dataset"]
).astype(int)

# only keep crypto site details for data centers that matched a crypto site
data_centers.loc[~matched_crypto_site, ["crypto_name", "crypto_company", "crypto_status"]] = np.nan

print("Data centers labeled crypto:", data_centers["is_crypto"].sum(), "of", len(data_centers))
print("Share labeled crypto:", round(data_centers["is_crypto"].mean() * 100, 1), "%")

In [ ]:
columns = [
    "facility_name", "operator_name", "state", "purpose",
    "crypto_from_distance", "crypto_from_address", "crypto_from_dc_dataset",
    "crypto_name", "crypto_miles"
]

data_centers.loc[data_centers["is_crypto"] == 1, columns].sort_values("crypto_miles")

##### How much does the cutoff matter?

The number of data centers matched by distance depends on `MAX_MILES`. This helps us choose it. (This table only counts the distance check, not the address or `purpose`/`facility_name` checks.)

In [ ]:
rows = []
for miles in [0.1, 0.25, 0.5, 1, 2]:
    labeled_dcs = data_centers[data_centers["crypto_miles"] <= miles]
    matched_crypto = crypto[crypto["nearest_dc_miles"] <= miles]

    rows.append({
        "max_miles": miles,
        "data_centers_labeled": len(labeled_dcs),
        "crypto_sites_matched": len(matched_crypto),
    })

pd.DataFrame(rows)

### EIA-860 Plant Data

The first row of each EIA-860 Excel file is a title, so the column names are on the second row (`header=1`).

In [ ]:
# Added header=1 to skip the first row

plants = pd.read_excel(DATA_DIR / "eia860_2025" / "2___Plant_Y2025.xlsx", header=1)
print("Rows and columns:", plants.shape)
plants.sample(3)

In [ ]:
plants.describe(include="all").T

### EIA-860 Generator Data

The generator file has three sheets: `Operable`, `Proposed`, and `Retired and Canceled`. For now we only look at `Operable`.

In [ ]:
generators = pd.read_excel(
    DATA_DIR / "eia860_2025" / "3_1_Generator_Y2025.xlsx",
    sheet_name="Operable",
    header=1
)
print("Rows and columns:", generators.shape)
generators.sample(3)

In [ ]:
generators.describe(include="all").T

EIA-860 Defining Energy Source Features

In [ ]:
energy_sources = pd.read_csv(
    DATA_DIR / "eia860_2025" / "eia_860_energy_source_codes.csv",
)
energy_sources.rename(columns={'Energy Source Code': 'Energy Source 1'}, inplace=True)
energy_sources.sample(3)

In [ ]:
#generators['Energy Source 1'].unique()
generator_features = generators[['State','County','Energy Source 1','Technology','Time from Cold Shutdown to Full Load','Summer Capacity (MW)','Winter Capacity (MW)']].copy()
# Dropping description to create categorical feature from Energy Source
generator_features_full = pd.merge(generator_features, energy_sources, on='Energy Source 1', how='left').drop('Description', axis=1)
generator_features_full.rename(columns={'Category':'Energy Category','Time from Cold Shutdown to Full Load':'Time to Full Load'}, inplace=True)
# Clean Time to Full Load from string to numerical value
generator_features_full.sample(15)

Time to Full Load in Minutes Clean Up

In [ ]:
def time_in_min(stringval):
    val = str(stringval).lower().strip()
    # Match digits versus text
    regex = re.match(r"(\d+)(\s*[a-z]+)", val)
    if not regex:
        return None

    if regex.group(2) in ['m']:
        return int(regex.group(1))
    elif regex.group(2) in ['h']:
        return int(regex.group(1))*60
    else:
        return None

In [ ]:
generator_features_full['Time to Full Load'] = generator_features_full['Time to Full Load'].apply(time_in_min)


In [ ]:
generator_features_full.describe()
generator_features_full['Time to Full Load'] = generator_features_full['Time to Full Load'].fillna(generator_features_full['Time to Full Load'].mean().round(1))

One-Hot Encoding for Energy

In [ ]:
one_hot_energy = pd.get_dummies(generator_features_full['Energy Source 1'])
generator_features_full = generator_features_full.join(one_hot_energy)
generator_features_full.sample(5)

### EIA-861M Retail Sales of Electricity


In [ ]:
# Added header=2 to skip the first two rows of the Excel sheet

sales = pd.read_excel(DATA_DIR / "eia861m_sales_revenue.xlsx", sheet_name="Monthly-States", header=2)
print("Rows and columns:", sales.shape)
sales.sample(3)

In [ ]:
sales.describe(include="all").T

In [ ]:
sales.info()

In [ ]:
sales['Year'] = sales['Year'].astype(str)

In [ ]:
# Looking at 2025 only and only .1 since that is Commercial
# EIA classifies data centers as commercial customers
sales_2025 = sales[sales['Year']=='2025'].copy()
sales_2025 = sales_2025[['Year','Month','State','Thousand Dollars.1','Megawatthours.1','Count.1','Cents/kWh.1']]
sales_2025.sample(5)

In [ ]:
# Group by State, Summer/Winter AVG
# Summer/Winter to match generator dataset capacity values
sales_2025['Season'] = sales_2025['Month'].apply(
    lambda x: 'summer' if 4 <= x <= 9 else 'winter'
)
grouped_sales = sales_2025.groupby(by=['State','Season'], as_index=False).mean(numeric_only=True).sort_values(by='State', ascending=True).drop(columns=['Month'])
grouped_sales.head(10)

### EIA-930 Balance Data

In [ ]:
# This is a subset of the EIA 930 Balance dataset for the second half of 2025

balance = pd.read_csv(DATA_DIR / "eia930" / "EIA930_BALANCE_2025_Jul_Dec.csv")
print("Rows and columns:", balance.shape)
balance.sample(3)

In [ ]:
balance.describe(include="all").T

### EIA-930 Subregion Data

In [ ]:
subregion = pd.read_csv(DATA_DIR / "eia930" / "EIA930_SUBREGION_2025_Jul_Dec.csv")
print("Rows and columns:", subregion.shape)
subregion.sample(3)

In [ ]:
subregion.describe(include="all").T

## Save cleaned data

Save the cleaned data centers, including the `is_crypto` label, to `data/processed/`. This file is the input to `03-supervised-learning.ipynb`.

In [ ]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

data_centers.to_csv(PROCESSED_DIR / "fractracker_data_centers_clean.csv", index=False)
print("Saved: fractracker_data_centers_clean.csv with", data_centers.shape[0], "rows and", data_centers.shape[1], "columns")